# MINI CLASSIFIER TO EXPLORE MODEL MODIFICATIONS: TRAIN

In [1]:
import os
import ast
import random
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms
import torchaudio.functional as AF

import soundfile as sf
import librosa
import timm

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split, StratifiedGroupKFold


## SELECT OLNY N = 4 CLASSES FROM DATA

In [2]:
# Select labesl
SELECTED_LABELS = ["whtdov", "undtin1", "compau", "trsowl"]

In [3]:
# Load tables and select
df_taxonomy = pd.read_csv("./data/taxonomy_mini.csv")
df_train = pd.read_csv("./data/train_mini.csv")
df_soundscapes = pd.read_csv("./data/train_soundscapes_labels_mini.csv") 

df_taxonomy = df_taxonomy[df_taxonomy["primary_label"].isin(SELECTED_LABELS)]
df_train = df_train[df_train["primary_label"].isin(SELECTED_LABELS)]

# Soundscapes
df_soundscapes["primary_label"] = (
    df_soundscapes["primary_label"]
    .astype(str)
    .apply(
        lambda x: ";".join(
            [
                label
                for label in x.split(";")
                if label in SELECTED_LABELS
            ]
        )
    )
)
df_soundscapes = df_soundscapes[
    df_soundscapes["primary_label"].apply(len) > 0
].reset_index(drop=True)

#  SAVE NEW TABLES
df_taxonomy.to_csv("./taxonomy_mini_ready.csv", index = False)
df_train.to_csv("./train_mini_ready.csv", index = False)
df_soundscapes.to_csv("./train_soundscapes_labels_mini_ready.csv", index = False)

# Load tables
df_taxonomy = pd.read_csv("./taxonomy_mini_ready.csv")
df_train = pd.read_csv("./train_mini_ready.csv")
df_soundscapes = pd.read_csv("./train_soundscapes_labels_mini_ready.csv") 

## UTILITY FUNCTIONS

In [4]:
# Dictionary
path_csv = os.path.expanduser("./taxonomy_mini_ready.csv")
df = pd.read_csv(path_csv)
primary_label = df["primary_label"].unique()

# Create label to target dict
n_classes = len(primary_label)
DICTIONARY_LABEL2TARGET = {}
for i, label in enumerate(primary_label):
    vec = np.zeros(n_classes) 
    vec[i] = 1
    DICTIONARY_LABEL2TARGET[label] = vec

# Create a dictionary label to sample weight (sampled by train audio only)
df_audio = pd.read_csv("./train_mini_ready.csv")
temp_dict = df_audio["primary_label"].value_counts().to_dict()
total_count = len(df_audio)

DICTIONARY_LABEL2WEIGHT = {}
for label, count in temp_dict.items():
    w = (count / total_count)
    DICTIONARY_LABEL2WEIGHT[label] = w

In [5]:
class UtilFunctions():
    '''
    Utility class for handling label preprocessing in multi-label
    audio classification tasks.

    Includes:
    - Multi-hot encoding generation
    - Label aggregation for soundscapes
    - Label smoothing for regularization
    '''

    def __init__(self):
        # Initialize dictionaries and number of classes
        self.label2target = DICTIONARY_LABEL2TARGET
        self.label2weight = DICTIONARY_LABEL2WEIGHT
        self.n_classes = len(self.label2target)

    def multi_hot_audios(self, primary_label, secondary_labels):
        # Start from primary label one-hot vector (copy to avoid mutation)
        vec_1 = self.label2target[primary_label].copy()

        # If no secondary labels, return primary vector
        if secondary_labels == "[]":
            return vec_1
        else:
            # Parse string representation of list into Python list
            s_labels = ast.literal_eval(secondary_labels)

            # Combine vectors via element-wise max (multi-hot encoding)
            for label in s_labels:
                if label in self.label2target:
                    vec_2 = self.label2target[label]
                    vec_1 = np.maximum(vec_1, vec_2)
            return vec_1
        
    def multi_hot_soundscapes(self, primary_label):
        # Split multiple labels separated by ';'
        p_labels = primary_label.split(";")

        # Initialize empty multi-hot vector
        vec_1 = np.zeros(self.n_classes)

        # Aggregate all label vectors into a single multi-hot vector
        for label in p_labels:
            vec_2 = self.label2target[label]
            vec_1 = np.maximum(vec_1, vec_2)
        return vec_1
    
    # LABEL SMOOTHING (For knowledge distilation)
    def label_smooth(self, y_true, mu = 0.05):
        y_true_smooth = y_true * (1 - mu)  +  (mu * torch.sum(y_true, dim=1, keepdim=True) / self.n_classes) 
        return y_true_smooth


class LossCombined(nn.Module):
    """Multilabel loss: combination of BCE and Focal Loss."""
    
    def __init__(self, beta = 0.3, rho = 2):
        super().__init__()
        self.beta = beta  # If beta=0.3: final_loss = 0.3 * BCE + 0.7 * Focal
        self.rho = rho    
        self.bce = nn.BCELoss(reduction = "none")   # reduction="none" -> do NOT average yet, we want per-element loss
        self.epsilon = 1e-7

    def loss_focal(self, y_pred, y_true):
        pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)    
        val = -((1 - pt) ** self.rho) * torch.log(pt)
        return val

    def forward(self, y_logits, y_true):
        
        # Transform logit to prob
        y_pred = torch.sigmoid(y_logits)
        y_pred = torch.clamp(y_pred, self.epsilon, 1-self.epsilon)

        # calculate BCE elementwise
        bce_loss = self.bce(y_pred, y_true)

        # calculate focal elementwise
        focal_loss = self.loss_focal(y_pred, y_true)

        # Combine both functions
        loss = self.beta * bce_loss + ((1-self.beta) * focal_loss)

        return loss.mean()


# PREPARE DATA

In [6]:
# path to main dataframe 
path_df = "./train_mini_ready.csv" 

# Load table with data info 
df = pd.read_csv(path_df) 
print(f"The dim of train table before transforming data is {df.shape}")
print(f"With columns: {list(df.columns)}")

# PREPROCESS DATA
# Delete specific columns
columns_deleted = ['common_name', "class_name", 'scientific_name', 
                   'inat_taxon_id', 'license', 'url', "latitude","longitude", "type"] 
df = df.drop(columns=columns_deleted).copy()

# Transform "collection" to one-hot for future partition and validation, and delete collection column
mask = (df["collection"] == "iNat")
df["iNat"] =  np.array(mask).astype(int)
df["XC"] = np.array(~mask).astype(int)
df = df.drop(columns="collection").copy()

# Load taxonomy table and count all classes (234)
path_taxonomy = "./taxonomy_mini_ready.csv"
df_taxonomy =  pd.read_csv(path_taxonomy)
primary_label = df_taxonomy["primary_label"].unique()
print(f"The total number of classes is {len(primary_label)}")

# Add new column "sample_weight" using precomputed dictionary
df["sample_weight"] = df["primary_label"].map(DICTIONARY_LABEL2WEIGHT)

# Print Final Dataframe to use
print(f"\nThe dim of table after transforming data is {df.shape}")
print(f"With columns: {list(df.columns)}")

# S P L I T 
# fill Missing values of author with "unknown"
df["author"] = df["author"].fillna("unknown").astype(str)

# Split data acording to y = "primary_label", and grouped by "autor"
folds = 5
cv = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=666)
df["fold"] = -1
for fold, (_, val_idx) in enumerate(cv.split(df, y=df["primary_label"], groups=df["author"])):
    df.loc[val_idx, "fold"] = fold

# Look how many vals per fold
print("Data per fold:")
print(df["fold"].value_counts().sort_index())


# Save dfs
path_to_save = "./train_folds_mini_ready.csv"
df.to_csv(path_to_save, index=False)
print(f"\nThe train dataframes were created and saved succesfully at {path_to_save}")
print(f"Number of features: {len(df.columns)-1}")
df

The dim of train table before transforming data is (1679, 15)
With columns: ['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude', 'scientific_name', 'common_name', 'class_name', 'inat_taxon_id', 'author', 'license', 'rating', 'url', 'filename', 'collection']
The total number of classes is 4

The dim of table after transforming data is (1679, 8)
With columns: ['primary_label', 'secondary_labels', 'author', 'rating', 'filename', 'iNat', 'XC', 'sample_weight']
Data per fold:
fold
0    309
1    318
2    349
3    406
4    297
Name: count, dtype: int64

The train dataframes were created and saved succesfully at ./train_folds_mini_ready.csv
Number of features: 8


,primary_label,secondary_labels,author,rating,filename,iNat,XC,sample_weight,fold
0,compau,[],Joe Klaiber,5.0,compau/XC254120.ogg,0,1,0.293627,4
1,compau,[],Aby Uribe,0.0,compau/iNat48550.ogg,1,0,0.293627,2
2,compau,[],lopezcor93,0.0,compau/iNat956023.ogg,1,0,0.293627,0
3,compau,[],Mayron McKewy Mejía,5.0,compau/XC247996.ogg,0,1,0.293627,1
4,compau,[],Mauricio Álvarez-Rebolledo (Colección de Sonid...,5.0,compau/XC525625.ogg,0,1,0.293627,4
...,...,...,...,...,...,...,...,...,...
1674,whtdov,[],Daniel Correia,0.0,whtdov/iNat901179.ogg,1,0,0.292436,1
1675,whtdov,"['undtin1', 'rubthr1']",Jeremy Minns,4.5,whtdov/XC229252.ogg,0,1,0.292436,3
1676,whtdov,[],Paul Marvin,4.0,whtdov/XC147512.ogg,0,1,0.292436,4
1677,whtdov,[],Edwin Calderón,4.0,whtdov/XC190507.ogg,0,1,0.292436,2


In [7]:
sample_rate = 32000
n_fft = 2048
hop_length = 512
n_mels = 256

mel = torchaudio.transforms.MelSpectrogram(sample_rate=sample_rate, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, power=2.0)
db = torchaudio.transforms.AmplitudeToDB()

total_sum = 0.0
total_sq_sum = 0.0
total_numel = 0

path_dir = os.path.expanduser("~/1_machine_learning_projects/BIRDS/data/train_audio")

for _, row in tqdm(df.iterrows(), total=len(df), desc="Computing stats"):

    file_path = os.path.join(path_dir, row["filename"])

    waveform, sr = sf.read(file_path)

    waveform = torch.tensor(waveform).float().unsqueeze(0)

    # pad si el clip es muy corto
    if waveform.shape[-1] < n_fft:
        waveform = torch.nn.functional.pad(
            waveform,
            (0, n_fft - waveform.shape[-1])
        )

    x = mel(waveform)
    x = db(x)

    total_sum += x.sum()
    total_sq_sum += (x ** 2).sum()
    total_numel += x.numel()

mean = total_sum / total_numel

std = torch.sqrt(
    total_sq_sum / total_numel - mean ** 2
)

print("mean:", mean.item())
print("std:", std.item())

Computing stats: 100%|████████████████████████████████████████████████████| 1679/1679 [03:52<00:00,  7.24it/s]

mean: -19.631881713867188
std: 18.53003692626953


## CREATE DATASETS

In [13]:
class BirdAudioDataset(Dataset):
    '''
    Inputs:
        - df: transformed df of "train.csv" containing data, target and k-Folds index
        - segment_seconds: final duration of audio in seconds after cutting it

    Outputs:
        - chunk: Tensor [audio sample]
        - target: Tensor [num_classes]
        - (optional) metadata: Tensor [6]

    Description:
    Loads audio, extracts fixed-length segment, applies augmentation,
    returns waveform and multi-label target.
    '''

    def __init__(self, df, segment_seconds = 5, p_noise = 0.5, p_filter = 0.25, p_mix = 0.66, p_soundscape_noise = 0.5, alpha = 0.3):
        self.df = df.copy()
        self.num_samples = len(self.df)
        self.segment_seconds = segment_seconds  # to cut audio in segments of 5s

        self.utils = UtilFunctions()
        
        self.path_audios = os.path.expanduser("~/1_machine_learning_projects/BIRDS/data/train_audio")
        self.audio_files = os.listdir(self.path_audios)
        
        self.path_soundscapes = os.path.expanduser("~/1_machine_learning_projects/BIRDS/data/train_soundscapes")
        self.soundscape_files = os.listdir(self.path_soundscapes)
        
        self.path_esc50 = os.path.expanduser("~/1_machine_learning_projects/BIRDS/data/train_esc50")
        self.esc50_files = os.listdir(self.path_esc50)

        self.p_noise = p_noise
        self.p_filter = p_filter
        self.p_mix = p_mix
        self.p_soundscape_noise = p_soundscape_noise
        self.alpha = alpha

    # Function to cut audio into "segment_seconds"
    def cut_audio(self, waveform, sample_rate=32000):
        # Verify if audio is larger than segment_seconds (5s)
        length_waveform = len(waveform)
        segment_samples = int(self.segment_seconds * sample_rate)
        
        if length_waveform <= segment_samples:
            # Add missing seconds as 0-values
            missing_segment = np.zeros( segment_samples - length_waveform, dtype = waveform.dtype)
            waveform = np.concatenate([waveform, missing_segment])      
        else:
            # Cut a 5s chunk randomly from waveform
            start = np.random.randint(0, (length_waveform - segment_samples)+1)
            end = start + segment_samples
            waveform = waveform[start:end]
        return waveform    

    # Function to add noise from train soundscapes or ESC50
    def random_add_noise(self, waveform):
        # Random trial to add noise
        if random.random() < self.p_noise:
            # Random trial to select source of noise
            if random.random() < self.p_soundscape_noise:
                # Create path to noise file
                path = self.path_soundscapes
                random_filename = random.choice(self.soundscape_files)
                path_filename = os.path.join(path, random_filename)

                # Load, cut and normalize noise
                noise, sample_rate = sf.read(path_filename)
                if noise.ndim == 2:
                    noise = noise.mean(axis=1)
                noise_chunk = self.cut_audio(noise, sample_rate)
            
            # Select noise
            else:
                path = self.path_esc50
                random_filename = random.choice(self.esc50_files)
                path_filename = os.path.join(path, random_filename)
                
                # Load, cut and normalize noise
                noise, sample_rate = sf.read(path_filename)
                if noise.ndim == 2:
                    noise = noise.mean(axis=1)
                if sample_rate != 32000:
                    noise = torch.tensor(noise, dtype=torch.float32)
                    noise = AF.resample(noise, sample_rate, 32000).numpy()
                    sample_rate = 32000
                noise_chunk = self.cut_audio(noise, sample_rate)
                
            # add noise to waveform
            waveform = waveform + (self.alpha * noise_chunk)
        return waveform

    # Function to Random filtering
    def random_filter(self, waveform, sample_rate=32000):
        if random.random() > self.p_filter:
            return waveform
        waveform = torch.tensor(waveform, dtype=torch.float32)
        waveform = AF.equalizer_biquad(waveform.unsqueeze(0), sample_rate=sample_rate, 
                        center_freq=random.uniform(500, 8000), gain=random.uniform(-6, 6), Q=random.uniform(0.5, 1.5)).squeeze(0)
        return waveform.numpy()
        

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Get rows
        row = self.df.iloc[idx]
        
        # Load audio filename and create path to file 
        filename = row["filename"]
        path_filename = os.path.join(self.path_audios, filename)

        # Extract waveform and 5s chunk
        waveform_1, _ = sf.read(path_filename)
        if waveform_1.ndim == 2:
            waveform_1 = waveform_1.mean(axis=1)
        chunk_1 = self.cut_audio(waveform_1)

        # Background mixing and Filtering
        chunk_1 = self.random_add_noise(chunk_1)
        chunk_1 = self.random_filter(chunk_1)

        # Get target
        target_1 = torch.tensor(self.utils.multi_hot_audios(row["primary_label"], row["secondary_labels"]), dtype=torch.float32)
 
        # Return chunk and target or randomly apply MixUP(add a second waveform to first one randomly)
        if random.random() > self.p_mix:
            # Noramlize and transform to tensor
            chunk_1 = torch.tensor(chunk_1, dtype = torch.float32)
            return chunk_1, target_1
        
        # MixUp
        idx_2 = random.randint(0, self.num_samples - 1)  
        row_2 = self.df.iloc[idx_2]
        waveform_2, _ = sf.read(os.path.join(self.path_audios, row_2["filename"]))
        if waveform_2.ndim == 2:
            waveform_2 = waveform_2.mean(axis=1)
        
        # Cut second waveform, randomly apply filter and noise
        chunk_2 = self.cut_audio(waveform_2)
        chunk_2 = self.random_add_noise(chunk_2)
        chunk_2 = self.random_filter(chunk_2)
        
        # Get second target
        target_2 = torch.tensor(self.utils.multi_hot_audios(row_2["primary_label"], row_2["secondary_labels"]),dtype=torch.float32)

        # Combine target and chunks
        target = torch.maximum(target_1, target_2)
        chunk = chunk_1 + chunk_2

        # Normalize chunk and transform to tensor:
        chunk = torch.tensor(chunk, dtype = torch.float32)

        return chunk, target

## MODEL

In [14]:
class BirdEfficientNetV2S(nn.Module):
    def __init__(self, num_classes=4, pretrained=True, dropout=0.25, n_fft=2048, hop_length=512, n_mels=128):

        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_mels = n_mels
        
        # CNN backbone
        self.backbone = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k", 
                        pretrained=pretrained, in_chans=1, num_classes=0, global_pool="avg")

        # Audio to Log-Mel
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=32000, n_fft=self.n_fft, 
                    hop_length=self.hop_length, n_mels=self.n_mels, power=2.0, f_min=20, normalized=True)
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)


        # Classification head
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, num_classes)
        )

    def forward(self, x):
        """
        Args = x: input tensor of shape [B, 32kHz * 5s]

        Returns =  y: raw class scores of shape [B, num_classes]
        """
        # Transform waveform to log-mel
        x = self.mel(x)  # [B,128,T]
        x = self.db(x)  
        
        # z Normalize
        global_mean = -19.631881713867188
        global_std = 18.53003692626953
        #mean = x.mean(dim=2, keepdim=True)  # media temporal por banda mel
        #std = x.std(dim=2, keepdim=True)    # desviación temporal por banda mel
        
        x = (x - global_mean) / (global_std + 1e-6)

        # CNN
        x = x.unsqueeze(1)  # [B,1,128,T]
        x = self.backbone(x)  # [B,feat_dim]
        # Head
        y = self.head(x)  # [B,234]

        return y

In [15]:
import torch
import torch.nn as nn
import timm


class BirdWignerNet(nn.Module):
    def __init__(
        self,
        num_classes=4,
        pretrained=True,
        dropout=0.25,
        sample_rate=32000,
        n_fft=1024,
        max_lag=256,
        hop=512,
        backbone_name="efficientnet_b0",
    ):
        super().__init__()

        # Parámetros de la representación Wigner-Ville
        self.sample_rate = sample_rate
        self.n_fft = n_fft
        self.max_lag = max_lag
        self.hop = hop

        # Backbone CNN más ligero que EfficientNetV2-S
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            in_chans=1,
            num_classes=0,
            global_pool="avg",
        )

        # Cabeza de clasificación
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, num_classes),
        )

    @torch.no_grad()
    def wigner_distribution(self, waveform):
        """
        Calcula Wigner-Ville sin construir grafo de autograd.

        Args:
            waveform: [B, L] o [L]

        Returns:
            W: [B, F, T]
        """

        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)

        s = waveform.to(torch.complex64)

        B, L = s.shape
        device = s.device

        times_idx = torch.arange(0, L, self.hop, device=device)

        T = len(times_idx)
        F = self.n_fft // 2 + 1

        W = torch.empty(B, T, F, device=device, dtype=torch.float32)

        for i, ti in enumerate(times_idx):
            ti = int(ti.item())

            tau_max = min(ti, L - 1 - ti, self.max_lag)

            local_corr = torch.zeros(
                B,
                self.n_fft,
                device=device,
                dtype=torch.complex64,
            )

            for tau in range(-tau_max, tau_max + 1):
                local_corr[:, tau % self.n_fft] = (
                    s[:, ti + tau] * torch.conj(s[:, ti - tau])
                )

            spec = torch.fft.fft(local_corr, dim=-1)

            W[:, i, :] = spec[:, :F].real

            del local_corr, spec

        W = W.permute(0, 2, 1)  # [B, F, T]

        return W

    @torch.no_grad()
    def preprocess_wigner(self, x):
        """
        Convierte waveform [B, L] a imagen [B, 1, F, T].
        No requiere gradientes.
        """

        x = self.wigner_distribution(x)

        # Compresión logarítmica con signo
        x = torch.sign(x) * torch.log1p(torch.abs(x))

        # Recorte para evitar outliers grandes
        x = torch.clamp(x, -5, 5)

        # Normalización por muestra
        mean = x.mean(dim=(1, 2), keepdim=True)
        std = x.std(dim=(1, 2), keepdim=True)

        x = (x - mean) / (std + 1e-6)

        # Canal para CNN: [B, 1, F, T]
        x = x.unsqueeze(1)

        return x

    def forward(self, x):
        """
        Args:
            x: waveform [B, samples]

        Returns:
            logits [B, num_classes]
        """

        # La Wigner no aprende, entonces no debe guardar grafo
        x = self.preprocess_wigner(x)

        # Solo el backbone + head tienen gradientes
        x = self.backbone(x)

        y = self.head(x)

        return y

# TRAIN PARAMETERS

In [16]:
import yaml

# Load config
with open("config_train.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# Train Hyperparameters
n_epochs = cfg["train"]["n_epochs"]
batch_size = cfg["train"]["batch_size"]
lr = cfg["train"]["lr"]
wd = cfg["train"]["wd"]

# Data Augmentation Config
gamma = cfg["augment"]["gamma"]
p_noise = cfg["augment"]["p_noise"]
p_mix = cfg["augment"]["p_mix"]
p_filter = cfg["augment"]["p_filter"]
p_soundscape_noise = cfg["augment"]["p_filter"]
alpha = cfg["augment"]["p_filter"]

# Model Config
dropout = cfg["model"]["dropout"]

# Spectrogram Config
n_fft = cfg["spec_balanced"]["n_fft"]
hop_length = cfg["spec_balanced"]["hop_length"]
n_mels = cfg["spec_balanced"]["n_mels"]
p_mask = cfg["spec_balanced"]["p_mask"]

# Loader Config
num_workers = cfg["loader"]["num_workers"]
pin_memory = cfg["loader"]["pin_memory"]
persistent_workers = cfg["loader"]["persistent_workers"]
prefetch_factor = cfg["loader"]["prefetch_factor"]

# Loss Config
beta = cfg["loss"]["beta"]
rho = cfg["loss"]["rho"]

# Label Smoothing
mu = cfg["mu"]

#device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"""
Training configuration
----------------------
Device              : {device}
Epochs              : {n_epochs}
Batch size          : {batch_size}
Gamma               : {gamma}
Noise probability   : {p_noise}
MixUp probability   : {p_mix}
Filter probability  : {p_filter}
Learning rate       : {lr}
Weight decay        : {wd}
\n
Model configuration
----------------------
n_fft       : {n_fft}
hop_length  : {hop_length}
n_mels      : {n_mels}
""")


Training configuration
----------------------
Device              : cuda
Epochs              : 20
Batch size          : 32
Gamma               : -0.5
Noise probability   : 0.5
MixUp probability   : 0.55
Filter probability  : 0.25
Learning rate       : 0.0001
Weight decay        : 0.0001


Model configuration
----------------------
n_fft       : 2048
hop_length  : 512
n_mels      : 128



# TRAIN

In [17]:
# Load dataframe 
path = os.path.expanduser("./train_folds_mini_ready.csv")
df = pd.read_csv(path)
n_folds = df["fold"].nunique()


# K = 5 Fold
#for fold in range(n_folds):
fold = 4
mel_configs = ["spec_rtemp"]
probs = [0.5]
for p in probs:
    p_soundscape_noise = p
    print(f"\n===== Fold {p} =====")
    # Spectrogram Config
    n_fft = cfg["spec_rtemp"]["n_fft"]
    hop_length = cfg["spec_rtemp"]["hop_length"]
    n_mels = cfg["spec_rtemp"]["n_mels"]
    p_mask = cfg["spec_rtemp"]["p_mask"]

    
    # Split train / validation folds
    df_train = df[df["fold"] != fold].reset_index(drop=True)
    df_val = df[df["fold"] == fold].reset_index(drop=True)
    
    # Create datasets
    data_train = BirdAudioDataset(df=df_train, p_noise=p_noise, p_filter=p_filter, p_mix=p_mix , p_soundscape_noise=p_soundscape_noise, alpha=alpha)
    data_val = BirdAudioDataset(df=df_val, p_noise=0.0, p_filter=0.0, p_mix=0.0)

    # Create train dataloader using sampler w load weights
    sample_weights = torch.tensor(df_train["sample_weight"].values ** gamma, dtype=torch.double) # Loader weights
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    loader_train = DataLoader(data_train, batch_size=batch_size, sampler=sampler, shuffle=False, 
                   num_workers=num_workers, pin_memory=pin_memory, persistent_workers=persistent_workers, prefetch_factor=prefetch_factor)

    # Create validation
    loader_val = DataLoader(data_val, batch_size=batch_size, shuffle=False, 
        num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)

    # Model
    model = BirdEfficientNetV2S(dropout=dropout, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels).to(device)
    #model = BirdWignerNet().to(device)

    # Loss + optimizer
    criterion = LossCombined(beta=beta, rho=rho)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-6)

    # Smooth y_true
    smooth_y = UtilFunctions() 

    # =========================== TRAINING LOOP =================================
    for epoch in range(n_epochs):
    
        # Put models in train mode
        model.train()
    
        # Create progress bar
        pbar = tqdm(loader_train, desc=f"Epoch {epoch+1}/{n_epochs}")
    
        for batch_X, batch_y in pbar:
                
            # To cpu
            batch_X = batch_X.to(device, non_blocking=True)
            batch_y = batch_y.float().to(device, non_blocking=True)
                
            # Reinitialize gradients
            optimizer.zero_grad(set_to_none=True)
    
            # Forward
            logits = model(batch_X)
    
            # Smooth target
            batch_y_s = smooth_y.label_smooth(batch_y)
    
            # Loss
            loss = criterion(logits, batch_y_s)
    
            # Backprop
            loss.backward()
            optimizer.step()
    
        # Scheduler
        scheduler.step()

        # ======================= VALIDATION LOOP ===============================
        model.eval()
        val_loss = 0.0

        with torch.no_grad():

            for batch_X, batch_y in loader_val:

                # To gpu
                batch_X = batch_X.to(device, non_blocking=True)
                batch_y = batch_y.float().to(device, non_blocking=True)

                # Forward
                logits = model(batch_X)

                # Validation loss
                loss = criterion(logits, batch_y)
                val_loss += loss.item()

        # Average validation losses
        #val_loss /= len(loader_val)

        # Print validation metrics
        print(
            f"[Fold {fold + 1}] "
            f"Epoch {epoch + 1}/{n_epochs} | "
            f"Val Loss  : {val_loss:.4f} | "
        )

        
    # Save Model
    p = int(p*100)
    torch.save(model.state_dict(), f"output/model_n_fft{n_fft}_hop_length{hop_length}_n_mels{n_mels}_p_soundscape_noise{p}_ultimate_global_mean.pt")
    print(f"Models saved at output/model_n_fft{n_fft}_hop_length{hop_length}_n_mels{n_mels}_p_soundscape_noise{p}_ultimate.pt")


===== Fold 0.5 =====


Epoch 1/20: 100%|█████████████████████████████████████████████████████████████| 44/44 [09:19<00:00, 12.73s/it]


[Fold 5] Epoch 1/20 | Val Loss  : 2.6922 | 


Epoch 2/20: 100%|█████████████████████████████████████████████████████████████| 44/44 [09:18<00:00, 12.69s/it]


[Fold 5] Epoch 2/20 | Val Loss  : 1.8677 | 


...

[Fold 5] Epoch 18/20 | Val Loss  : 1.1176 | 


Epoch 19/20: 100%|████████████████████████████████████████████████████████████| 44/44 [04:27<00:00,  6.07s/it]


[Fold 5] Epoch 19/20 | Val Loss  : 1.0907 | 


Epoch 20/20: 100%|████████████████████████████████████████████████████████████| 44/44 [04:25<00:00,  6.03s/it]


[Fold 5] Epoch 20/20 | Val Loss  : 1.0852 | 
Models saved at output/model_n_fft2048_hop_length512_n_mels256_p_soundscape_noise50_ultimate.pt
